# Lab 2 — pandas I: Load, Explore and Clean — **SOLUTION**
**TunisMarket** `orders.csv`: 12,060 raw rows → trusted clean dataset, every decision documented.
Run locally (file in `data/orders.csv`) or in Colab (upload `orders.csv` → `/content`).

In [1]:
import os
import pandas as pd

pd.options.display.max_columns = 20
pd.options.display.float_format = "{:,.2f}".format
print("pandas", pd.__version__)
PATH = "data/orders.csv" if os.path.exists("data/orders.csv") else "/content/orders.csv"

pandas 3.0.5


## Exercise 1 — Load and first look

In [2]:
orders = pd.read_csv(PATH)
display(orders.head(10))
print("Shape:", orders.shape)          # (12060, 10)
orders.info()

,order_id,customer_id,product_id,order_date,quantity,unit_price,discount_pct,city,payment_method,status
0,O07078,C0592,P022,2025-05-04,1.00,23.82,0,Monastir,Card,Delivered
1,O05946,C0883,P014,2025-02-19,1.00,45.65,0,Tunis,Cash on delivery,Returned
2,O08635,C1378,P018,2025-07-23,1.00,39.81,0,Ben Arous,Card,Delivered
3,O08493,C0335,P021,2025-07-17,2.00,55.18,0,Tunis,Card,Delivered
4,O02535,C0421,P038,2024-07-20,2.00,12.23,0,Sfax,Card,Delivered
5,O01525,C1178,P036,2024-05-10,3.00,18.34,0,Bizerte,Cash on delivery,Delivered
6,O08940,C0926,P005,2025-08-09,2.00,9.21,0,Nabeul,Card,Delivered
7,O04467,C1297,P022,2024-11-24,1.00,24.13,0,Sousse,Wallet,Delivered
8,O02316,C0653,P039,2024-07-06,1.00,35.37,0,Nabeul,Cash on delivery,Delivered
9,O01718,C0643,P033,2024-05-27,2.00,9.10,10,Nabeul,Cash on delivery,Delivered


Shape: (12060, 10)
<class 'pandas.DataFrame'>
RangeIndex: 12060 entries, 0 to 12059
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        12060 non-null  str    
 1   customer_id     12060 non-null  str    
 2   product_id      12060 non-null  str    
 3   order_date      12040 non-null  str    
 4   quantity        12060 non-null  float64
 5   unit_price      11915 non-null  float64
 6   discount_pct    12060 non-null  int64  
 7   city            12060 non-null  str    
 8   payment_method  11819 non-null  str    
 9   status          12060 non-null  str    
dtypes: float64(2), int64(1), str(7)
memory usage: 942.3 KB


In [3]:
orders.describe()
# Suspicious: (1) quantity max = 999 (impossible), (2) missing values in order_date, unit_price, payment_method
# (3) quantity is float64 instead of int

,quantity,unit_price,discount_pct
count,"12,060.00","11,915.00","12,060.00"
mean,2.87,44.41,3.02
std,35.17,32.79,5.47
min,1.00,8.73,0.00
25%,1.00,22.09,0.00
50%,1.00,35.64,0.00
75%,2.00,53.90,5.00
max,999.00,153.45,20.00


## Exercise 2 — Select and filter

In [4]:
display(orders[["order_id", "city", "status"]].head())

mask = (orders["status"] == "Delivered") & (orders["city"] == "Tunis") & (orders["quantity"] >= 3)
with_loc = orders.loc[mask, ["order_id", "city", "quantity", "unit_price"]]
with_query = orders.query("status == 'Delivered' and city == 'Tunis' and quantity >= 3")[["order_id", "city", "quantity", "unit_price"]]
print(len(with_loc), len(with_query), "-> same:", len(with_loc) == len(with_query))

,order_id,city,status
0,O07078,Monastir,Delivered
1,O05946,Tunis,Returned
2,O08635,Ben Arous,Delivered
3,O08493,Tunis,Delivered
4,O02535,Sfax,Delivered


460 460 -> same: True


## Exercise 3 — Missing values

In [5]:
missing_pct = (orders.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.round(2))
orders["payment_method"] = orders["payment_method"].fillna("Unknown")
# Not 0 everywhere: 0 is a fake value -> a price of 0 or a date of 0 does not exist and it biases the averages.

payment_method   2.00
unit_price       1.20
order_date       0.17
order_id         0.00
customer_id      0.00
product_id       0.00
quantity         0.00
discount_pct     0.00
city             0.00
status           0.00
dtype: float64


## Exercise 4 — Duplicates

In [6]:
print("Duplicates:", orders.duplicated().sum())    # 60
before = len(orders)
orders = orders.drop_duplicates()
print(before, "->", len(orders))                   # 12060 -> 12000

Duplicates: 60
12060 -> 12000


## Exercise 5 — Text cleaning and dates

In [7]:
print(orders["city"].nunique(), "spellings before")         # 50
orders["city"] = orders["city"].str.strip().str.title()
print(orders["city"].nunique(), "cities after:", sorted(orders["city"].unique()))   # 10

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")  # bad dates -> NaT
orders = orders.dropna(subset=["order_date"])
print(orders["order_date"].dtype, len(orders))             # datetime64  11980

50 spellings before
10 cities after: ['Ariana', 'Ben Arous', 'Bizerte', 'Gabès', 'Kairouan', 'Monastir', 'Nabeul', 'Sfax', 'Sousse', 'Tunis']
datetime64[us] 11980


## Exercise 6 — Outliers and imputation

In [8]:
print(orders["quantity"].value_counts().sort_index())
orders = orders[orders["quantity"] <= 10].copy()              # business rule: max 10 units

median_price = orders.groupby("product_id")["unit_price"].transform("median")
orders["unit_price"] = orders["unit_price"].fillna(median_price)
orders["quantity"] = orders["quantity"].astype("int64")

print(orders[["unit_price", "quantity", "order_date"]].isna().sum())
print("Max quantity:", orders["quantity"].max(), "| dtype:", orders["quantity"].dtype)

quantity
1.00      7455
2.00      2613
3.00      1076
4.00       470
5.00       351
999.00      15
Name: count, dtype: int64
unit_price    0
quantity      0
order_date    0
dtype: int64
Max quantity: 5 | dtype: int64


## Exercise 7 — Export and verify

In [9]:
orders.to_csv("orders_clean_lab2.csv", index=False)
orders.head(500).to_excel("orders_sample.xlsx", index=False)

check = pd.read_csv("orders_clean_lab2.csv", parse_dates=["order_date"])
print("Same shape:", check.shape == orders.shape, "| order_date:", check["order_date"].dtype)

raw_rows, clean_rows = 12060, len(orders)
print(f"Raw rows: {raw_rows} | Clean rows: {clean_rows} | Removed: {(raw_rows - clean_rows) / raw_rows * 100:.1f} %")

Same shape: True | order_date: datetime64[us]
Raw rows: 12060 | Clean rows: 11965 | Removed: 0.8 %


## The same pipeline as ONE reusable function (slide 11)

In [10]:
def clean_orders(raw):
    df = raw.drop_duplicates().copy()
    df["city"] = df["city"].str.strip().str.title()
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df = df.dropna(subset=["order_date"])
    df = df[df["quantity"] <= 10].copy()
    df["payment_method"] = df["payment_method"].fillna("Unknown")
    df["unit_price"] = df["unit_price"].fillna(df.groupby("product_id")["unit_price"].transform("median"))
    df["quantity"] = df["quantity"].astype("int64")
    return df

clean = clean_orders(pd.read_csv(PATH))
assert len(clean) == 11965 and clean.isna().sum().sum() == 0
print("clean_orders() ->", clean.shape)

clean_orders() -> (11965, 10)
